In [ ]:
"""
Notebook 03: Data Cleaning and Feature Engineering
====================================================

This notebook prepares the Kepler KOI dataset for modeling.

Pipeline:
1. Load the relevant subset (output of Notebook 02).
2. Filter to binary classification (CONFIRMED vs FALSE POSITIVE).
3. Handle missing values.
4. Detect and handle outliers.
5. Engineer new features.
6. Encode the target variable.
7. Save the final dataset to data/processed/.

Input:  data/interim/kepler_koi_relevant.csv
Output: data/processed/kepler_clean.csv

Author: Mersen-cloud
"""

# Standard library
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display configuration
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Plot styling
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print("Imports successful ✓")

In [ ]:
# Define paths
PROJECT_ROOT = Path.cwd().parent
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

INPUT_FILE = DATA_INTERIM / "kepler_koi_relevant.csv"
OUTPUT_FILE = DATA_PROCESSED / "kepler_clean.csv"

# Load dataset
df = pd.read_csv(INPUT_FILE)
print(f"✓ Loaded {INPUT_FILE.name}")
print(f"  Initial shape: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Filter to binary classification: CONFIRMED vs FALSE POSITIVE
# CANDIDATEs are removed because their true class is unknown — using them
# would introduce label noise into the model.

print("Disposition counts BEFORE filtering:")
print(df['koi_disposition'].value_counts())
print()

df_binary = df[df['koi_disposition'].isin(['CONFIRMED', 'FALSE POSITIVE'])].copy()

print("Disposition counts AFTER filtering:")
print(df_binary['koi_disposition'].value_counts())
print(f"\nShape after filter: {df_binary.shape}")
print(f"Rows removed (CANDIDATE): {len(df) - len(df_binary)}")

In [ ]:
# Missing values in the binary subset
missing = df_binary.isnull().sum()
missing_pct = (missing / len(df_binary) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

missing_df = missing_df[missing_df['missing_count'] > 0]
print("Missing values in binary subset:")
print(missing_df)

In [ ]:
# Drop columns that won't be used for modeling
columns_to_drop = [
    'kepoi_name',      # Identifier, not a feature
    'kepler_name',     # Mostly NaN (only confirmed have names)
    'koi_score'        # Derived from disposition itself (data leakage risk)
]

df_clean = df_binary.drop(columns=columns_to_drop)
print(f"✓ Dropped {len(columns_to_drop)} non-feature columns")
print(f"  Remaining shape: {df_clean.shape}")
print(f"  Columns: {list(df_clean.columns)}")

In [ ]:
# Identify numerical columns with NaN
numerical_cols_with_nan = df_clean.select_dtypes(include=[np.number]).columns
nan_cols = [col for col in numerical_cols_with_nan if df_clean[col].isnull().any()]

print(f"Columns with NaN to impute: {len(nan_cols)}")
for col in nan_cols:
    median = df_clean[col].median()
    n_missing = df_clean[col].isnull().sum()
    df_clean[col] = df_clean[col].fillna(median)
    print(f"  {col}: filled {n_missing} NaN with median = {median:.4f}")

# Verify no NaN remain
print(f"\nRemaining NaN after imputation: {df_clean.isnull().sum().sum()}")

In [ ]:
# Outlier detection using IQR rule
def detect_outliers_iqr(series):
    """Return a boolean mask indicating outliers using the IQR rule."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Apply to key physical variables
physical_cols = ['koi_period', 'koi_prad', 'koi_teq', 'koi_depth', 'koi_duration']

outlier_summary = []
for col in physical_cols:
    mask = detect_outliers_iqr(df_clean[col])
    outlier_summary.append({
        'column': col,
        'n_outliers': mask.sum(),
        'pct_outliers': round(mask.sum() / len(df_clean) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
print("Outliers detected (IQR method):")
print(outlier_df)

In [ ]:
# Apply log transformation to highly skewed variables
skewed_cols = ['koi_period', 'koi_depth', 'koi_insol', 'koi_prad']

for col in skewed_cols:
    new_col = f'log_{col}'
    df_clean[new_col] = np.log1p(df_clean[col])
    print(f"  Created {new_col} from {col}")

print(f"\nNew shape after adding log features: {df_clean.shape}")

In [ ]:
# Compare original vs log-transformed distributions
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i, col in enumerate(skewed_cols):
    # Original
    axes[0, i].hist(df_clean[col], bins=60, color='steelblue', edgecolor='black')
    axes[0, i].set_title(f'{col} (original)', fontweight='bold')
    axes[0, i].set_ylabel('Count')
    
    # Log-transformed
    axes[1, i].hist(df_clean[f'log_{col}'], bins=60, color='seagreen', edgecolor='black')
    axes[1, i].set_title(f'log_{col}', fontweight='bold')
    axes[1, i].set_ylabel('Count')

plt.suptitle('Effect of Log Transformation on Skewed Variables', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Create physically meaningful derived features

# Planet-to-star radius ratio — directly related to transit depth
df_clean['planet_star_radius_ratio'] = df_clean['koi_prad'] / df_clean['koi_srad']

# Temperature ratio — how hot the planet is relative to its star
df_clean['planet_star_temp_ratio'] = df_clean['koi_teq'] / df_clean['koi_steff']

# Sum of all false positive flags — a strong indicator of being a FP
fp_flags = ['koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec']
df_clean['fp_flag_sum'] = df_clean[fp_flags].sum(axis=1)

print("✓ New features created:")
print("  - planet_star_radius_ratio")
print("  - planet_star_temp_ratio")
print("  - fp_flag_sum")
print(f"\nFinal shape: {df_clean.shape}")

In [ ]:
# Encode target variable: CONFIRMED → 1, FALSE POSITIVE → 0
df_clean['target'] = (df_clean['koi_disposition'] == 'CONFIRMED').astype(int)

# Verify encoding
print("Target distribution:")
print(df_clean['target'].value_counts())
print(f"\nProportion of positive class (CONFIRMED): {df_clean['target'].mean():.2%}")

In [ ]:
# Final dataset overview
print("=" * 60)
print("FINAL CLEAN DATASET")
print("=" * 60)
print(f"Shape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")
print(f"\nDtypes:\n{df_clean.dtypes}")
print(f"\nNaN check: {df_clean.isnull().sum().sum()} total NaN")

In [ ]:
# Save the cleaned and feature-engineered dataset
df_clean.to_csv(OUTPUT_FILE, index=False)
print(f"✓ Clean dataset saved to: {OUTPUT_FILE}")
print(f"  File size: {OUTPUT_FILE.stat().st_size / 1024:.1f} KB")
print(f"  Ready for modeling 🚀")

## 📌 Resumen del pipeline de limpieza

### Decisiones tomadas

**Filtrado a clasificacion binaria**
- Eliminamos objetos CANDIDATE (1.978 filas) porque su clase verdadera no se conoce.
- Quedamos con 7.586 filas: 2.747 CONFIRMED + 4.839 FALSE POSITIVE.

**Columnas eliminadas**
- `kepoi_name`, `kepler_name`: identificadores, no predictivos.
- `koi_score`: derivada de la propia disposicion, riesgo de data leakage.

**Imputacion de NaN**
- ~363 filas tenian NaN en variables fisicas.
- Estrategia: imputar con la mediana (robusta a outliers).

**Outliers**
- Detectados con regla IQR, pero NO eliminados.
- Razon: en astronomia los valores extremos suelen ser reales.
- Solucion: transformacion logaritmica para comprimir el rango.

### Features creadas

| Feature | Calculo | Significado |
|---|---|---|
| log_koi_period | log1p(periodo) | Periodo en escala log |
| log_koi_depth | log1p(depth) | Profundidad de transito en escala log |
| log_koi_insol | log1p(insol) | Flujo de radiacion en escala log |
| log_koi_prad | log1p(prad) | Radio planetario en escala log |
| planet_star_radius_ratio | prad / srad | Ratio fisico clave para transitos |
| planet_star_temp_ratio | teq / steff | Cuan caliente esta el planeta vs su estrella |
| fp_flag_sum | suma de 4 flags | Indicador agregado de falso positivo |

### Variable objetivo
- target = 1: CONFIRMED (exoplaneta confirmado)
- target = 0: FALSE POSITIVE
- Proporcion positiva: ~36%

### Proximos pasos (Notebook 04)
- Train/test split.
- Modelos baseline: Logistic Regression y Random Forest.
- Evaluacion con metricas multiples.